# Embeddings

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Let's see how embeddings work on a few examples.

We'll start with a query:

In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [3]:
v1.shape

(384,)

Encode our document:

In [4]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

Next, we compare the query against the document using dot product:

In [5]:
v1.dot(dv)

np.float32(0.323324)

Now we try an unrelated query:

In [6]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

This time the similarity with the document should be much smaller:

In [7]:
v2.dot(dv)

np.float32(0.019730454)

The first score for `q1` vs `d` (0.32) is higher, so that query is more similar to the document about registration. The second score for `q2` vs `d` sits near 0, because installing Docker has nothing to do with registration. A score near 0 means the two vectors are about as different as they can be.

That's the whole idea behind vector search: similar texts get similar vectors, and a dot product tells us how similar.

# Embedding Our Dataset

## Loading the data

Our Jupyter notebook (`vector_search.ipynb`) is running inside the `02-vector-search` directory, but the file we want to import (`ingest.py`) is located in a sibling directory, `01-agentic-rag`.

By default, Python only looks for modules in the current working directory of the notebook.

Here is how we fix this directly inside our notebook: Adding the Directory to Python's Path
—We need to explicitly tell Python to look inside the `01-agentic-rag` folder before it tries to import `ingest`.

In [8]:
import sys
import os

# Get the absolute path to the sibling directory
sibling_dir = os.path.abspath('../01-agentic-rag')

# Add it to sys.path so Python knows to look there
if sibling_dir not in sys.path:
    sys.path.append(sibling_dir)

In [9]:
from ingest import load_faq_data

documents = load_faq_data()

## Generating embeddings

In [10]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [11]:
texts[10]

'Course: How many hours per week am I expected to spend on this course? It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'

In [12]:
len(texts)

1350

In [13]:
from tqdm.auto import tqdm

Next we chunk the dataset into batches of 50 and encode each batch:

In [14]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [15]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

We end up with 1350 vectors.

We turn them into a 2-dimensional array (matrix) where

- rows are documents (vectors)
- columns are dimensions of the vectors

In [16]:
import numpy as np
X = np.array(vectors)

In [17]:
X.shape

(1350, 384)

# Vector Search

## Scoring documents

We have a matrix `X` with all document embeddings. We take a query, compare it against every document, and keep the most similar ones.

When a query comes in, we embed it:

Next, we compute the dot product against all documents:

In [18]:
scores = X.dot(v1)

In [19]:
scores

array([ 0.4874059 ,  0.20991927,  0.7629411 , ..., -0.08637965,
        0.03759794, -0.03037034], shape=(1350,), dtype=float32)

## Best match

In [20]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.7629411))

In [21]:
documents[2]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

## Top 5 results

In [22]:
top5 = np.argsort(scores)[-5:]

In [23]:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [24]:
scores[top5]

array([0.7629411 , 0.7579372 , 0.7192131 , 0.6536312 , 0.56009996],
      dtype=float32)

In [25]:
top5 = np.argsort(-scores)[:5]

In [26]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629411
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579372
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related

# Vector Search with minsearch

In the previous section we did vector search by hand with numpy. We embedded the query, computed dot products, and found the best matches. Writing the argsort and matrix code every time gets old, and it can't filter by course. So instead we'll use a library that wraps all of it.

We'll use minsearch, the small in-memory search library we already used in module 1 for text search. It has a `VectorSearch` class for vector search.

Both classes share the same API:

- `fit` to index data
- `search` to query
- `filter_dict` in `search` to filter by keyword

It's the simplest way to get started with vector search.

## Creating the index
We already have our documents and vectors from the previous section.

Index them:

In [27]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

We pass the numpy array `X` with all embeddings and the list of documents as payload. The `keyword_fields` parameter works the same as in the text `Index`, so we can filter by course later.

## Searching

In [28]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

Under the hood it does the same thing we just did by hand. It computes the dot product between each vector (after filtering) and our query vector.

Look at the top result:

In [29]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

## Filtering by course

Like the text index, we can filter by keyword fields. This matters for user experience. A student in LLM Zoom Camp doesn't care about answers from the data engineering course. So we narrow to their course first, then score only within it.

Pass a `filter_dict`:

In [30]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [31]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

Now that we can run vector search, let's use it in RAG.

# RAG with Vector Search

In module 1, we built a RAG pipeline with three steps:

```python
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)
```

The search step used keyword search. Now we swap in vector search. Because RAG is modular, search is the only step we touch. Build prompt and the LLM call stay exactly as they were.

## Using RAGBase

First, create the OpenAI client:

In [32]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

Next, download and index the data:

In [33]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

Then use the RAGBase class:

In [34]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

Ask it a question:

In [35]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.'

This still uses keyword search. Text search isn't bad here, so the answer may already look right. Next we replace search with vector search.

We already have:

- All the indexed documents `documents`
- The embeddings matrix `X` with all these documents
- The vector search engine `vindex`

We can't pass `vindex` to RAG as-is. Text search takes the query string directly, but vector search needs the query as a vector first. So we subclass `RAGBase` and override `search` to encode the query before searching.

The subclass overrides `search`:

In [36]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

The `__init__` method adds one extra argument, `embedder`, for the sentence transformer. Inside `search` we use it to turn the query into a vector. Then we query `vindex` with that vector instead of the raw text. Everything else is inherited from `RAGBase`.

## Using it

In [37]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [38]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join. You can start learning and submitting homework while the form is open. If you want a certificate, make sure you submit your project while submissions are still being accepted.'

# Vector Search with sqlitesearch

## Creating the index

In [39]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

## Indexing the data

In [40]:
vs_index.fit(vectors, documents)

## Searching

Search works the same way as with minsearch. We always encode the query into a vector first. This is one thing that makes vector search heavier than text search. With text search we'd throw the raw query straight at the engine.

Encode, then search:

In [41]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [42]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

## Filtering by course

In [43]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [44]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

## Closing the connection

In [45]:
vs_index.close()

## Reopening the index

In a new Python session, you can reopen the index without re-computing embeddings:

In [46]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Now we can search:

In [47]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

We still load the embedding model to encode the query, but we don't re-embed all the documents. No `fit` call needed, because the index is already built and waiting on disk.

This is the same two-process split we used for text search in module 1. One process ingests and builds the index, another queries it.

It matters more here than with text search. Embedding the whole dataset takes about a minute. We don't want a user waiting that long when the app starts up. We pay that cost once during ingestion, and the query side starts up instantly.

## Using sqlitesearch vector search in RAG

Let's use our persistent vector index in RAG.

Create a new notebook: `vector_search_persistent.ipynb`

In a new notebook, set up the model and open the index (same as the "Reopening the index" section above)